# 🏢💀 Bankrupt-AI — przewidywanie bankructwa polskich firm
## Część 1: Dane, opis i eksploracja

**Autor:** Wojciech Płonka (projekt indywidualny)

**Przedmiot:** Uczenie maszynowe w Python — laboratorium

---

W tej części:
1. pobieramy dane o polskich firmach,
2. opisujemy co zawierają,
3. sprawdzamy braki i czyścimy dane,
4. wizualizujemy zbiór przed trenowaniem modeli.

> Modele trenujemy w części 2 (`02_modele.ipynb`).

## Krok 0 — Importy

Wczytujemy biblioteki. `pandas` to tabele danych, `numpy` to liczby, `matplotlib`/`seaborn` to wykresy, a `scipy.io.arff` posłuży do odczytu pliku danych (format `.arff`, w którym UCI udostępnia ten zbiór).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import urllib.request, zipfile, io, os
from scipy.io import arff

sns.set_theme(style='whitegrid')
print('Biblioteki wczytane ✅')

## Krok 1 — Pobranie danych

Dane pochodzą z **UCI Machine Learning Repository** (zbiór nr 365: *Polish companies bankruptcy data*).

Zbiór składa się z 5 plików `.arff` (`1year` … `5year`). Każdy plik to firmy obserwowane na N lat przed
potencjalnym bankructwem. Pobieramy ZIP-a, rozpakowujemy i **łączymy wszystkie 5 plików w jedną dużą tabelę**
(dodajemy kolumnę `forecast_year` — z którego pliku pochodzi wiersz).

In [ ]:
# adres zbioru danych w repozytorium UCI
URL = 'https://archive.ics.uci.edu/static/public/365/polish+companies+bankruptcy+data.zip'

# pobieramy plik ZIP do pamięci
print('Pobieram dane z UCI...')
with urllib.request.urlopen(URL) as resp:
    zip_bytes = resp.read()
print(f'Pobrano {len(zip_bytes)/1024:.0f} KB')

# rozpakowujemy ZIP i czytamy wszystkie pliki .arff
ramki = []
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    pliki_arff = sorted([n for n in z.namelist() if n.endswith('.arff')])
    print('Pliki w archiwum:', pliki_arff)
    for nazwa in pliki_arff:
        with z.open(nazwa) as f:
            dane, meta = arff.loadarff(io.TextIOWrapper(f, encoding='utf-8'))
        ramka = pd.DataFrame(dane)
        # numer roku z nazwy pliku, np. '3year.arff' -> 3
        ramka['forecast_year'] = int(nazwa.replace('year.arff', '').split('/')[-1])
        ramki.append(ramka)

# sklejamy wszystkie ramki w jedną
df = pd.concat(ramki, ignore_index=True)
print('\nGotowe ✅  Łączny rozmiar zbioru:', df.shape)

## Krok 2 — Pierwszy rzut oka na dane

Sprawdzamy jak wygląda tabela: pierwsze wiersze, rozmiar, nazwy kolumn i podstawowe statystyki.

In [ ]:
df.head()

In [ ]:
print('Liczba wierszy (firm):  ', df.shape[0])
print('Liczba kolumn:          ', df.shape[1])
print('\nNazwy kolumn:')
print(list(df.columns))

### Poprawka etykiety `class`

W formacie `.arff` kolumna `class` wczytuje się jako **bajty** (`b'0'` / `b'1'`) zamiast zwykłych liczb.
Zamieniamy ją na liczbę całkowitą:
- **0** = firma przetrwała (zdrowa),
- **1** = firma zbankrutowała.

In [ ]:
# z bajtów b'0'/b'1' robimy liczbę 0/1
df['class'] = df['class'].apply(lambda x: int(x.decode()) if isinstance(x, bytes) else int(x))
df['class'].value_counts()

## Krok 2.5 — Opis danych (co oznaczają kolumny)

Kolumny nazywają się `Attr1` … `Attr64` — to **wskaźniki finansowe** zdefiniowane w dokumentacji UCI.
Poniżej oficjalny słownik (przyda się do opisu danych w sprawozdaniu i do interpretacji wyników).

In [ ]:
opis_wskaznikow = {
    'Attr1': 'zysk netto / aktywa ogółem',
    'Attr2': 'zobowiązania ogółem / aktywa ogółem',
    'Attr3': 'kapitał obrotowy / aktywa ogółem',
    'Attr4': 'aktywa obrotowe / zobowiązania krótkoterminowe',
    'Attr5': '[(gotówka + pap. wart. + należności - zob. krótkot.) / (koszty operac. - amortyzacja)] * 365',
    'Attr6': 'zyski zatrzymane / aktywa ogółem',
    'Attr7': 'EBIT / aktywa ogółem',
    'Attr8': 'wartość księgowa kapitału własnego / zobowiązania ogółem',
    'Attr9': 'sprzedaż / aktywa ogółem',
    'Attr10': 'kapitał własny / aktywa ogółem',
    'Attr11': '(zysk brutto + poz. nadzw. + koszty finansowe) / aktywa ogółem',
    'Attr12': 'zysk brutto / zobowiązania krótkoterminowe',
    'Attr13': '(zysk brutto + amortyzacja) / sprzedaż',
    'Attr14': '(zysk brutto + odsetki) / aktywa ogółem',
    'Attr15': '(zobowiązania ogółem * 365) / (zysk brutto + amortyzacja)',
    'Attr16': '(zysk brutto + amortyzacja) / zobowiązania ogółem',
    'Attr17': 'aktywa ogółem / zobowiązania ogółem',
    'Attr18': 'zysk brutto / aktywa ogółem',
    'Attr19': 'zysk brutto / sprzedaż',
    'Attr20': '(zapasy * 365) / sprzedaż',
    'Attr21': 'sprzedaż (rok n) / sprzedaż (rok n-1)',
    'Attr22': 'zysk operacyjny / aktywa ogółem',
    'Attr23': 'zysk netto / sprzedaż',
    'Attr24': 'zysk brutto (w 3 lata) / aktywa ogółem',
    'Attr25': '(kapitał własny - kapitał akcyjny) / aktywa ogółem',
    'Attr26': '(zysk netto + amortyzacja) / zobowiązania ogółem',
    'Attr27': 'zysk operacyjny / koszty finansowe',
    'Attr28': 'kapitał obrotowy / aktywa trwałe',
    'Attr29': 'logarytm aktywów ogółem',
    'Attr30': '(zobowiązania ogółem - gotówka) / sprzedaż',
    'Attr31': '(zysk brutto + odsetki) / sprzedaż',
    'Attr32': '(zob. krótkoterminowe * 365) / koszt sprzedanych produktów',
    'Attr33': 'koszty operacyjne / zobowiązania krótkoterminowe',
    'Attr34': 'koszty operacyjne / zobowiązania ogółem',
    'Attr35': 'zysk ze sprzedaży / aktywa ogółem',
    'Attr36': 'sprzedaż ogółem / aktywa ogółem',
    'Attr37': '(aktywa obrotowe - zapasy) / zobowiązania długoterminowe',
    'Attr38': 'kapitał stały / aktywa ogółem',
    'Attr39': 'zysk ze sprzedaży / sprzedaż',
    'Attr40': '(aktywa obrotowe - zapasy - należności) / zob. krótkoterminowe',
    'Attr41': 'zobowiązania ogółem / ((zysk operac. + amortyzacja) * (12/365))',
    'Attr42': 'zysk operacyjny / sprzedaż',
    'Attr43': 'rotacja należności + rotacja zapasów (w dniach)',
    'Attr44': '(należności * 365) / sprzedaż',
    'Attr45': 'zysk netto / zapasy',
    'Attr46': '(aktywa obrotowe - zapasy) / zobowiązania krótkoterminowe',
    'Attr47': '(zapasy * 365) / koszt sprzedanych produktów',
    'Attr48': 'EBITDA (zysk operac. - amortyzacja) / aktywa ogółem',
    'Attr49': 'EBITDA / sprzedaż',
    'Attr50': 'aktywa obrotowe / zobowiązania ogółem',
    'Attr51': 'zobowiązania krótkoterminowe / aktywa ogółem',
    'Attr52': '(zob. krótkoterminowe * 365) / koszt sprzedanych produktów',
    'Attr53': 'kapitał własny / aktywa trwałe',
    'Attr54': 'kapitał stały / aktywa trwałe',
    'Attr55': 'kapitał obrotowy',
    'Attr56': '(sprzedaż - koszt sprzedanych produktów) / sprzedaż',
    'Attr57': '(aktywa obrotowe - zapasy - zob. krótkot.) / (sprzedaż - zysk brutto - amortyzacja)',
    'Attr58': 'koszty ogółem / sprzedaż ogółem',
    'Attr59': 'zobowiązania długoterminowe / kapitał własny',
    'Attr60': 'sprzedaż / zapasy',
    'Attr61': 'sprzedaż / należności',
    'Attr62': '(zob. krótkoterminowe * 365) / sprzedaż',
    'Attr63': 'sprzedaż / zobowiązania krótkoterminowe',
    'Attr64': 'sprzedaż / aktywa trwałe',
}
print(f'Słownik zawiera opis {len(opis_wskaznikow)} wskaźników finansowych.')
# podgląd kilku pierwszych
for k in list(opis_wskaznikow)[:5]:
    print(f'  {k}: {opis_wskaznikow[k]}')

In [ ]:
# podstawowe statystyki kilku przykładowych wskaźników
df[['Attr1', 'Attr2', 'Attr3', 'Attr10', 'forecast_year']].describe()

## Krok 3 — Braki danych (czyszczenie)

Realne dane finansowe mają **braki** (firma nie podała jakiegoś wskaźnika). Sprawdźmy ile ich jest.

In [ ]:
# ile braków w każdej kolumnie (TOP 15 najbardziej dziurawych)
braki = df.isna().sum().sort_values(ascending=False)
print('Kolumny z największą liczbą braków:')
print(braki.head(15))
print(f'\nŁączny procent brakujących wartości w całym zbiorze: {df.isna().mean().mean()*100:.2f}%')

### Strategia uzupełnienia braków

Najbardziej dziurawa kolumna to `Attr37`. Braków jest sporo, więc **nie usuwamy wierszy** (stracilibyśmy
dużą część danych). Zamiast tego każdą brakującą wartość uzupełniamy **medianą** danej kolumny
(mediana jest odporna na wartości skrajne — typowe dla danych finansowych).

> To dokładnie ta sama logika, co uzupełnianie wieku średnią w zadaniu z Titanikiem na zajęciach —
> tylko tu wybieramy medianę, bo wskaźniki finansowe mają bardzo duże „odstające" wartości.

In [ ]:
# kolumny ze wskaźnikami (wszystkie poza etykietą i rokiem)
kolumny_cech = [c for c in df.columns if c.startswith('Attr')]

# uzupełniamy braki medianą każdej kolumny
df[kolumny_cech] = df[kolumny_cech].fillna(df[kolumny_cech].median())

print('Braki po uzupełnieniu:', df[kolumny_cech].isna().sum().sum(), '✅')

## Krok 4 — Wizualizacja danych

### 4.1 Jak bardzo dane są niezbalansowane?

To kluczowa cecha tego zbioru: bankrutów jest **bardzo mało**. Zobaczmy proporcję.

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Liczba firm: zdrowe vs bankruci')
plt.xlabel('')
plt.ylabel('Liczba firm')
plt.xticks([0,1], ['Zdrowa (0)', 'Bankrut (1)'])
# podpisy z liczbami nad słupkami
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.show()

udzial = df['class'].mean()*100
print(f'Bankruci stanowią tylko {udzial:.1f}% wszystkich firm.')
print('=> To zbiór NIEZBALANSOWANY — trzeba to uwzględnić przy ocenie modeli (sama accuracy zwiedzie!).')

### 4.2 Bankructwa w zależności od horyzontu (ile lat przed)

Pliki `1year`…`5year` to różne odległości czasowe od momentu bankructwa.

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x='forecast_year', hue='class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Liczba firm wg horyzontu prognozy')
plt.xlabel('Plik danych (lata przed potencjalnym bankructwem)')
plt.ylabel('Liczba firm')
plt.legend(title='', labels=['Zdrowa', 'Bankrut'])
plt.show()

### 4.3 Mapa ciepła korelacji (przykładowe wskaźniki)

64 wskaźniki na jednej mapie byłyby nieczytelne, więc bierzemy kilkanaście reprezentatywnych
(rentowność, zadłużenie, płynność) i sprawdzamy jak się ze sobą wiążą oraz z bankructwem.

In [ ]:
wybrane = ['Attr1','Attr2','Attr3','Attr6','Attr7','Attr10','Attr16','Attr17','Attr46','class']
plt.figure(figsize=(10,8))
sns.heatmap(df[wybrane].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Mapa ciepła korelacji wybranych wskaźników')
plt.show()

### 4.4 Rozkład wybranego wskaźnika: zdrowe vs bankruci

`Attr2` = zobowiązania / aktywa (poziom zadłużenia). Sprawdźmy czy bankruci faktycznie są bardziej zadłużeni.

In [ ]:
plt.figure(figsize=(8,4))
# obcinamy skrajne 1% ogonów, żeby wykres był czytelny
dane = df[(df['Attr2'] > df['Attr2'].quantile(0.01)) & (df['Attr2'] < df['Attr2'].quantile(0.99))]
sns.kdeplot(data=dane, x='Attr2', hue='class', fill=True, palette=['#2ecc71', '#e74c3c'])
plt.title('Rozkład zadłużenia (Attr2 = zobowiązania / aktywa)')
plt.xlabel('zobowiązania / aktywa')
plt.show()

## Krok 5 — Zapis przygotowanych danych

Zapisujemy wyczyszczony zbiór do pliku CSV, żeby w części 2 (modele) wczytać go od razu — bez ponownego pobierania i czyszczenia.

In [ ]:
df.to_csv('dane_oczyszczone.csv', index=False)
print('Zapisano: dane_oczyszczone.csv', df.shape, '✅')
# w Colabie: plik pojawi się po lewej w panelu plików (ikona folderu).
# Można go pobrać:  from google.colab import files; files.download('dane_oczyszczone.csv')

---
## ✅ Podsumowanie części 1

- Pobraliśmy realne dane o polskich firmach (5 plików scalonych w jeden zbiór).
- Opisaliśmy 64 wskaźniki finansowe.
- Naprawiliśmy etykietę i uzupełniliśmy braki medianą.
- Zobaczyliśmy, że zbiór jest mocno **niezbalansowany** (mało bankrutów) — to wpłynie na dobór i ocenę modeli.

**Następny krok:** `02_modele.ipynb` — trenowanie i porównanie modeli (KNN, drzewo, las losowy, boosting).